# Dialogue State Tracking Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: rule-based slot extractor

See `code/main.py`. Regex + synonym dictionaries cover 70% of canonical utterances in narrow domains:

In [ ]:
```python

CUISINE_SYNONYMS = {

    "italian": ["italian", "pasta", "pizza", "italy"],

    "chinese": ["chinese", "chow mein", "noodles"],

}

def extract_cuisine(utterance):

    for canonical, synonyms in CUISINE_SYNONYMS.items():

        if any(syn in utterance.lower() for syn in synonyms):

            return canonical

    return None

In [ ]:
```

Brittle outside the canonical vocabulary. Works for deterministic slot confirmations.

### Step 2: state update loop

In [ ]:
```python

def update_state(state, utterance):

    new_state = dict(state)

    for slot, extractor in SLOT_EXTRACTORS.items():

        value = extractor(utterance)

        if value is not None:

            new_state[slot] = value

    for slot in NEGATION_CLEARS:

        if is_negated(utterance, slot):

            new_state[slot] = None

    return new_state

In [ ]:
```

Three invariants:

- Never reset a slot the user did not touch.

- Explicit negation ("never mind the cuisine") must clear.

- User correction ("actually...") must overwrite, not append.

### Step 3: LLM-driven DST with structured output

In [ ]:
```python

from pydantic import BaseModel

from typing import Literal, Optional

import instructor

class RestaurantState(BaseModel):

    cuisine: Optional[Literal["italian", "chinese", "indian", "thai", "any"]] = None

    area: Optional[Literal["north", "south", "east", "west", "center"]] = None

    price: Optional[Literal["cheap", "moderate", "expensive"]] = None

    people: Optional[int] = None

    day: Optional[str] = None

def llm_dst(history, llm):

    prompt = f"""You track the slot values of a restaurant booking across turns.

Dialogue so far:

{render(history)}

Update the state based on the latest user turn. Output only the JSON state."""

    return llm(prompt, response_model=RestaurantState)

In [ ]:
```

Instructor + Pydantic guarantees a valid state object. No regex, no schema mismatches, no hallucinated slots.

### Step 4: JGA evaluation

In [ ]:
```python

def joint_goal_accuracy(predicted_states, gold_states):

    correct = sum(1 for p, g in zip(predicted_states, gold_states) if p == g)

    return correct / len(predicted_states)

In [ ]:
```

Calibrate: what fraction of turns does the system get ALL slots right? For MultiWOZ 2.4, top 2026 systems: 80-83%. Your in-domain system should exceed that on your narrow vocabulary or the LLM baseline beats you.

### Step 5: handling correction

In [ ]:
```python

CORRECTION_CUES = {"actually", "no wait", "on second thought", "change that to"}

def is_correction(utterance):

    return any(cue in utterance.lower() for cue in CORRECTION_CUES)

In [ ]:
```

On a detected correction, overwrite the last-updated slot rather than appending. Hard to get right without LLM help. The modern pattern: always let the LLM regenerate the whole state from history rather than incrementally updating — this naturally handles corrections.

## Exercises

In [ ]:
1. **Easy.** Build the rule-based state tracker in `code/main.py` for 3 slots (cuisine, area, price). Test on 10 hand-crafted dialogues. Measure JGA.
2. **Medium.** Same dataset with Instructor + Pydantic + a small LLM. Compare JGA. Inspect the hardest turns.
3. **Hard.** Implement both and route: rule-based primary, LLM fallback when rule-based emits <2 slots with confidence. Measure the combined JGA and inference cost per turn.